# Crawling

## Overview

In this portion of the homework, you will crawl Google Jobs to collect job descriptions for later cleaning (HW-3.2 and HW-3.3) and EDA (HW-3.4).

We will use the [SerpApi](https://serpapi.com/) service to query Google Jobs. SerpApi is a paid API, but its free tier is more than enough for this homework. The API lets you search Google programmatically, which has a wealth of practical applications.

You will need an API key from SerpApi. It is free to sign up, and you should not enter any payment information.

https://serpapi.com/manage-api-key

At the time of writing, the free plan gives you **250 searches per month**, with a throughput cap of **50 searches per hour**. Each Google Jobs search returns up to 10 job descriptions.

This portion of the homework relies on limited API search resources, so prototype carefully. Once you are sure your code is working 100%, run it one last time for your final "production" crawl.

Budget for this assignment:

* About 10 searches for prototyping and debugging
* About 90 searches for the production run (30 job titles x 3 pages)
* The remaining searches are a safety buffer. Do not burn them on careless re-runs.

Make sure to save the outputs, and back them up (e.g., commit them to your GitHub repo) after the final run, in case you delete them by mistake.

We will use SerpApi's official Python client, which can be installed with

`pip install serpapi`

`Note`: older tutorials use the legacy `google-search-results` package (`from serpapi import GoogleSearch`). Both packages install a module named `serpapi`, so do not install both in the same environment. If you previously installed the legacy package, run `pip uninstall google-search-results` first.

The following are additional useful reference resources

* [https://serpapi.com/google-jobs-api](https://serpapi.com/google-jobs-api): parameters and response format for the Google Jobs engine
* [https://serpapi.com/integrations/python](https://serpapi.com/integrations/python): the Python client
* [https://serpapi.com/pricing](https://serpapi.com/pricing): current free-tier limits

An example of a saved search result is provided in `data/example-out-AI-Solutions-Architect-2024-10-20-15-41-05.json`. Open it to see the structure of the response before you spend any searches.

## Starter code

Here is some starter code:

`Note: uule parameter`

The `uule` parameter is an encoded location parameter used in Google search queries. It encodes the geographic location from which the search is being conducted, which influences the results to be more relevant to that location.

We set it to `'w+CAIQICINVW5pdGVkIFN0YXRlcw'`, which is an encoded string representing the United States. This simulates searches as if they are being conducted from that location.

SerpApi also accepts a human-readable `location` parameter (e.g., `'location': 'United States'`) instead. Use one or the other, not both.

In [1]:
import csv
import hashlib
import html
import json
import os
import re
import time
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

# Default execution is offline: reopening or running the notebook costs no searches.
RUN_LIVE = os.environ.get("HW3_RUN_LIVE", "0") == "1"
STAGE = os.environ.get("HW3_STAGE", "all")  # trial, second, or all
DATA_DIR = Path("data")
REPORT_DIR = Path("reports")
LEDGER_FILE = REPORT_DIR / "search-attempts.json"
KEYS_FILE = Path(os.environ.get("HW3_KEYS_FILE", "~/.api-keys.json")).expanduser()
REQUEST_INTERVAL = 75  # conservative fallback; live account check adjusts this safely
MAX_SEARCH_ATTEMPTS = 100  # all recorded trials + production requests, not per run
client = None
API_KEY = None
print("Mode:", "LIVE" if RUN_LIVE else "OFFLINE (no API requests)")

Mode: LIVE


Save your API key in a centralized location outside of your project folder, e.g. `~/.api-keys.json`, and read it in with `json`. The file might look like:

```json
{
    "serpapi": "your-serpapi-key",
    "openai": "your-openai-key"
}
```

`IMPORTANT`: Never paste API keys directly into a notebook, never print them, and never commit them to GitHub. Keys that end up in a notebook output cell or a commit are effectively public.

In [2]:
if RUN_LIVE:
    import serpapi  # Official package: pip install serpapi

    if Path.cwd().resolve() in KEYS_FILE.resolve().parents:
        raise ValueError("Keep the API key file outside this repository.")
    if not KEYS_FILE.is_file():
        raise FileNotFoundError("Create the external JSON key file before a live crawl.")
    API_KEY = json.loads(KEYS_FILE.read_text(encoding="utf-8")).get("serpapi")
    if not isinstance(API_KEY, str) or not API_KEY.strip():
        raise ValueError("The external key file needs a nonempty 'serpapi' value.")
    API_KEY = API_KEY.strip()
    client = serpapi.Client(api_key=API_KEY, timeout=60)
    print("SerpApi client ready; the key is never displayed.")
else:
    print("Offline preview: credentials are not read and no client is created.")

SerpApi client ready; the key is never displayed.


Optional: SerpApi's account endpoint is free and does not count against your search quota, so you can use it to check how many searches you have left at any time. (Don't print the whole response, it includes your API key.)

In [3]:
# The account endpoint is free. Never display or save the whole response.
if RUN_LIVE:
    try:
        account = client.account()
    except Exception:
        raise RuntimeError("Account check failed; verify the key and connection.") from None
    safe_account = {
        name: account.get(name)
        for name in ("total_searches_left", "account_rate_limit_per_hour", "this_hour_searches")
    }
    print(safe_account)
    remaining = safe_account["total_searches_left"]
    if not isinstance(remaining, (int, float)) or remaining < 1:
        raise RuntimeError("No confirmed search quota remains; the crawl was not started.")
    hourly_limit = safe_account["account_rate_limit_per_hour"]
    if isinstance(hourly_limit, (int, float)) and hourly_limit > 0:
        # Use the live account limit, with a two-second safety margin.
        REQUEST_INTERVAL = max(15, int(3600 / hourly_limit) + 3)
        print("Minimum request interval (seconds):", REQUEST_INTERVAL)
        used = safe_account["this_hour_searches"] or 0
        if used >= hourly_limit:
            raise RuntimeError("Hourly quota is already exhausted. Resume later.")
    # Do not retain account details or the API key in saved notebook outputs.
    del account
else:
    print("Account check skipped in offline mode.")

{'total_searches_left': 244, 'account_rate_limit_per_hour': 250, 'this_hour_searches': 6}
Minimum request interval (seconds): 17


Be careful, don't run the following cells too many times while debugging and prototyping, or you will use up your free searches.

In [4]:
search_query = "data science"
params = {
    "engine": "google_jobs",
    "q": search_query,
    "uule": "w+CAIQICINVW5pdGVkIFN0YXRlcw",
    "hl": "en",
    "gl": "us",
}

### Inspect the provided example without using a search

The starter demonstration below reads the instructor's 2024 example. It is used only to understand the response structure, and is excluded from the production counts. Live requests are centralized in the utility function to enforce caching, pacing, and a shared request budget.

In [5]:
example_file = DATA_DIR / "example-out-AI-Solutions-Architect-2024-10-20-15-41-05.json"
result_dict = json.loads(example_file.read_text(encoding="utf-8"))
print("Source: instructor-provided example, not a new search")
print("Search query:", result_dict.get("search_parameters", {}).get("q"))
print("Example records:", len(result_dict.get("jobs_results", [])))

Source: instructor-provided example, not a new search
Search query: AI Solutions Architect
Example records: 10


In [6]:
for result in result_dict.get("jobs_results", [])[:5]:
    print(result.get("title", "Unknown title"), ":", result.get("company_name", "Unknown company"))

Senior Solutions Architect Architect, AI Foundation Model Providers, Digital Native Business : Amazon Web Services (AWS)
Artificial Intelligence (AI) Solutions Architect : Amivero
AI Solution Architect : Seekr Technologies
Sr. Specialist Solutions Architect, Generative AI & ML : Amazon Web Services (AWS)
AI Solutions Architect : Kind Lending, LLC


In web crawling, pagination involves retrieving data across multiple pages of results. It helps manage large datasets by fetching a limited number of results per request, enabling efficient data extraction without overwhelming the server or exceeding resource limits.

You can use pagination to get more results for a given search, however you need to start where the last search left off.

You do this by adding the `next_page_token` to the `params` dictionary.

You get the latest `next_page_token` from `result_dict["serpapi_pagination"]`. If that key is missing, there are no more pages for that query.

`Note`: Google Jobs no longer supports the older `start` (offset) or `num` parameters. Pagination is done only with `next_page_token`, and each page returns at most 10 results.

If you don't paginate and search 10 times, you will just get the same first 10 results over and over again.

In [7]:
example_token = result_dict.get("serpapi_pagination", {}).get("next_page_token")
print("Next page available:", bool(example_token))  # Do not dump tokens or URLs.

Next page available: True


Build a paginated request only if a next-page token exists. This preview makes no request and does not reuse the old example token in the live crawl.

In [8]:
pagination_preview = dict(params)
if example_token:
    pagination_preview["next_page_token"] = example_token
print("Pagination parameter included:", "next_page_token" in pagination_preview)

Pagination parameter included: True


In [9]:
print("Live pagination is handled by crawl_titles below; this example costs 0 searches.")

Live pagination is handled by crawl_titles below; this example costs 0 searches.


In [10]:
print("Fields available in the example:", sorted(result_dict.get("jobs_results", [{}])[0]))

Fields available in the example: ['apply_options', 'company_name', 'description', 'detected_extensions', 'extensions', 'job_highlights', 'job_id', 'location', 'share_link', 'thumbnail', 'title', 'via']


# Utility function

Create a utility function that searches Google Jobs and saves the results to a file.

Here is one sketch of what the function might look like:

- Imports the current date and time using `datetime`.
- Defines `search_google_jobs` to perform a Google Jobs search with a default or custom query.
- Accepts parameters for the search query, pagination token, and verbosity.
- Sets search parameters like location, language, and search engine.
- Appends the pagination token if provided.
- Creates a timestamped output filename based on the query and time, e.g. `data/machine-learning-engineer-2026-10-20-15-04-53.json`.
- Makes sure the output directory exists (`os.makedirs("data", exist_ok=True)`).
- Does a search and data extraction, with error handling.
- Optionally prints a short summary if `verbose` is `True` (job titles and companies, NOT the full params dict, which contains your API key) and saves the results to a JSON file.
- Returns the `next_page_token` for pagination, or `None` if there are no more pages or an error occurred.

In [11]:
def slugify(text):
    """Make query text safe for portable filenames."""
    ascii_text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "-", ascii_text.lower()).strip("-")[:90] or "jobs"


def save_json(path, value):
    """Atomically save one JSON file so interrupted writes cannot look complete."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)


def sanitize(value):
    """Remove credential fields and redact the active key, including in URLs."""
    if isinstance(value, dict):
        return {
            key: sanitize(item) for key, item in value.items()
            if key.lower() not in {"api_key", "apikey", "api-key", "access_token", "authorization"}
        }
    if isinstance(value, list):
        return [sanitize(item) for item in value]
    if isinstance(value, str):
        if API_KEY:
            value = value.replace(API_KEY, "[REDACTED]")
        return re.sub(r"(?i)([?&](?:api_key|access_token)=)[^&\s]+", r"\1[REDACTED]", value)
    return value


def request_params(query, token=None):
    parameters = {
        "engine": "google_jobs", "q": query,
        "uule": "w+CAIQICINVW5pdGVkIFN0YXRlcw", "hl": "en", "gl": "us",
    }
    if token:
        parameters["next_page_token"] = token
    return parameters


def fingerprint(parameters):
    return hashlib.sha256(json.dumps(parameters, sort_keys=True).encode()).hexdigest()


def saved_pages():
    """Only include this crawl's files; exclude the instructor's example."""
    pages = []
    for path in sorted(DATA_DIR.glob("*.json")):
        if path.name.startswith("example-out-"):
            continue
        # Invalid saved JSON is an error, never silently treated as a cache miss.
        value = json.loads(path.read_text(encoding="utf-8"))
        if "crawl_metadata" in value:
            pages.append((path, value))
    return pages


def load_attempts():
    return json.loads(LEDGER_FILE.read_text(encoding="utf-8")) if LEDGER_FILE.exists() else []


def search_google_jobs(search_query="data scientist", next_page_token=None, verbose=False, page=1):
    """Save a response and return its next token, or None for exhaustion/error.

    Reuse a saved query/token response before considering any network request.
    Failed searches are not retried automatically. Their type/status is recorded
    separately in reports/ so data/*.json contains only successful crawl pages.
    """
    parameters = request_params(search_query, next_page_token)
    request_id = fingerprint(parameters)
    for path, cached in saved_pages():
        if cached["crawl_metadata"]["request_fingerprint"] == request_id:
            if verbose:
                print(f"Cached: {search_query}, page {page}, {len(cached.get('jobs_results', []))} records")
            if not cached.get("jobs_results"):
                return None
            return cached.get("serpapi_pagination", {}).get("next_page_token") or None
    if client is None:
        raise RuntimeError("A live client is required for a cache miss.")

    attempts = load_attempts()
    if len(attempts) >= MAX_SEARCH_ATTEMPTS:
        raise RuntimeError("The 100-attempt homework budget has been reached.")
    if attempts:
        delay = max(0, attempts[-1]["started_at_epoch"] + REQUEST_INTERVAL - time.time())
        if delay:
            print(f"Rate limit: waiting {delay:.0f} seconds before the next search.", flush=True)
            time.sleep(delay)
    started = datetime.now(timezone.utc)
    event = {
        "query": search_query, "page": page, "request_fingerprint": request_id,
        "started_at_epoch": time.time(), "status": "started",
    }
    attempts.append(event)
    # Reserve the attempt before making the request; restarts share this budget.
    save_json(LEDGER_FILE, attempts)
    try:
        response = client.search(parameters)
        result = response.as_dict() if hasattr(response, "as_dict") else dict(response)
    except Exception as exc:
        status_code = getattr(exc, "status_code", None)
        event.update(status="request_error", error_type=type(exc).__name__)
        save_json(LEDGER_FILE, attempts)
        print(f"Request failed for {search_query}, page {page}: {type(exc).__name__}; no automatic retry.")
        # Stop globally for authentication or quota errors rather than waste searches.
        if status_code in (401, 403, 429):
            raise RuntimeError(f"Search stopped: HTTP {status_code}; check credentials/quota.") from None
        return None
    if result.get("error") or result.get("search_metadata", {}).get("status") == "Error":
        event.update(status="api_error")
        save_json(LEDGER_FILE, attempts)
        print(f"API returned an error for {search_query}, page {page}; moving to the next title.")
        return None
    if result.get("search_metadata", {}).get("status") != "Success":
        event.update(status="invalid_response")
        save_json(LEDGER_FILE, attempts)
        print(f"Missing successful search metadata for {search_query}; moving to the next title.")
        return None
    jobs = result.get("jobs_results", [])
    if not isinstance(jobs, list) or any(not isinstance(job, dict) for job in jobs):
        event.update(status="invalid_response")
        save_json(LEDGER_FILE, attempts)
        print(f"Invalid job records for {search_query}, page {page}; moving to the next title.")
        return None
    # An empty successful page is still saved as evidence of exhausted results.
    result["crawl_metadata"] = {
        "query": search_query, "page": page, "collected_at_utc": started.isoformat(),
        "request_fingerprint": request_id, "job_count": len(jobs),
    }
    filename = f"{slugify(search_query)}-{started:%Y-%m-%d-%H-%M-%S-%f}-p{page}.json"
    path = DATA_DIR / filename
    save_json(path, sanitize(result))
    event.update(status="saved", file=str(path), job_count=len(jobs))
    save_json(LEDGER_FILE, attempts)
    if verbose:
        print(f"Saved: {search_query}, page {page}: {len(jobs)} records -> {path}", flush=True)
        for job in jobs[:3]:
            print("  ", job.get("title", "Unknown"), "|", job.get("company_name", "Unknown"))
    if not jobs:
        return None
    return result.get("serpapi_pagination", {}).get("next_page_token") or None


def crawl_titles(titles, pages_per_title=3):
    """Resume each pagination chain; stop a title on failure or exhaustion."""
    if pages_per_title not in (1, 2, 3):
        raise ValueError("Use between 1 and 3 pages per title.")
    for query in titles:
        token = None
        seen = set()
        for page in range(1, pages_per_title + 1):
            next_token = search_google_jobs(query, token, verbose=True, page=page)
            if not next_token:
                break
            if next_token in seen:
                print(f"Repeated pagination token for {query}; stopping this title.")
                break
            seen.add(next_token)
            token = next_token


def crawl_summary(titles):
    """Report collection coverage without cleaning/deduplicating the raw data."""
    pages = saved_pages()
    attempts = load_attempts()
    rows = []
    for query in titles:
        values = sorted(
            [value for _, value in pages if value["crawl_metadata"]["query"] == query],
            key=lambda value: value["crawl_metadata"]["page"],
        )
        status = "not_started"
        if values:
            last = values[-1]
            finished = (last["crawl_metadata"]["page"] >= 3 or not last.get("jobs_results")
                        or not last.get("serpapi_pagination", {}).get("next_page_token"))
            status = "complete_3_pages" if len(values) == 3 else "exhausted" if finished else "incomplete"
            tokens = [v.get("serpapi_pagination", {}).get("next_page_token") for v in values]
            nonempty_tokens = [t for t in tokens if t]
            if len(nonempty_tokens) != len(set(nonempty_tokens)):
                status = "pagination_loop"
        query_attempts = [item for item in attempts if item["query"] == query]
        if query_attempts and query_attempts[-1]["status"] != "saved":
            status = query_attempts[-1]["status"]
        rows.append({
            "query": query, "pages_saved": len(values),
            "raw_records": sum(len(value.get("jobs_results", [])) for value in values),
            "status": status,
        })
    return rows

In [12]:
# The utility is exercised with fake responses in tests/test_crawl.py first.
# This avoids spending credits just to demonstrate a function call.
print("Example safe filename:", slugify("NLP Engineer (Natural Language Processing)"))
print("Utility ready; real requests occur only in the live collection cell.")

Example safe filename: nlp-engineer-natural-language-processing
Utility ready; real requests occur only in the live collection cell.


In [13]:
print("Test command: python -m unittest discover -s tests -v")

Test command: python -m unittest discover -s tests -v


In [14]:
print("Pagination uses next_page_token; neither 'start' nor 'num' is sent.")

Pagination uses next_page_token; neither 'start' nor 'num' is sent.


Read back the most recent file your function saved, to confirm that it was written correctly.

In [15]:
production_files = saved_pages()
if production_files:
    latest_file, saved = max(production_files, key=lambda item: item[1]["crawl_metadata"]["collected_at_utc"])
    print("Latest production file:", latest_file)
    print("Records read back:", len(saved.get("jobs_results", [])))
    for result in saved.get("jobs_results", [])[:5]:
        print(result.get("title", "Unknown"), ":", result.get("company_name", "Unknown"))
else:
    print("No live crawl files yet. The example above is not counted as collected data.")

Latest production file: data/machine-learning-engineer-2026-10-30-21-10-44-200438-p3.json
Records read back: 10
Machine Learning Engineer, Scenes : Waymo
Senior Machine Learning Engineer – NLP/LLM : USA018 Thomson Reuters Holdings Inc.
Engineering Manager - Machine Learning Engineering : The Aerospace Corp
Machine Learning Engineer : THE DOW CHEMICAL COMPANY
Audio Machine Learning Intern/Co-op : Bose Corporation


`Checkpoint`: this is a good point to commit your work (`git add`, `git commit -m "..."`), if you haven't already.

## Iterate over job titles

The following 30 titles reflect a wide range of roles that leverage data science and machine learning skills across industries and specialties. You may swap a few of them for titles that interest you more, but keep the total at about 30 so that you stay within your search budget.

In [16]:
job_titles = [
    "Data Scientist",
    "Machine Learning Engineer",
    "Artificial Intelligence Specialist",
    "Data Analyst",
    "Business Intelligence Analyst",
    "Research Scientist (AI-ML)",
    "Deep Learning Engineer",
    "NLP Engineer (Natural Language Processing)",
    "Computer Vision Engineer",
    "Data Engineer",
    "Applied Scientist",
    "Quantitative Analyst (Quant)",
    "AI Solutions Architect",
    "Statistician",
    "Big Data Engineer",
    "Data Science Consultant",
    "Automation Engineer",
    "Analytics Manager",
    "Operations Research Analyst",
    "Robotics Engineer",
    "Bioinformatics Data Scientist",
    "Financial Data Scientist",
    "Customer Insights Analyst",
    "Marketing Data Analyst",
    "Data Strategy Manager",
    "Cloud AI Engineer",
    "Computational Scientist",
    "Fraud Detection Specialist",
    "Risk Analyst",
    "Data Architect",
]

print(len(job_titles), "titles x 3 pages =", len(job_titles) * 3, "searches")

30 titles x 3 pages = 90 searches


For each job title, do three searches using pagination. This will result in around 30 jobs per title (assuming at least 30 jobs exist for that title). Save each search result to its own file.

`Rate limits`: the free plan allows at most 50 searches per hour, and the full crawl is about 90 searches. Either

* pace your requests, e.g. `time.sleep(75)` between searches (the full crawl then takes about 2 hours, so let it run in the background), or
* split the job titles into batches of at most 15 titles (45 searches), and run one batch per hour.

Also make your loop robust: if a search fails or returns no `next_page_token`, move on to the next title instead of crashing, and skip titles whose results are already saved in `data/` so that a re-run does not repeat searches you already paid for.

Now insert code to iterate over the job titles and perform the searches.

Be very careful, this needs to be 100% correct before running it, otherwise you will burn through your free searches.

I recommend doing just one iteration of the loop as a trial run. If that looks good, do the next iteration and carefully check the results. If everything looks good, then run the remaining 28 iterations.

Note: sometimes pagination returns fewer than 10 results, so you may end up with slightly fewer than 30 results per title, e.g. 25 to 30.

Remember to clean the job titles before using them in filenames, i.e. remove or replace characters like spaces, `/`, or `()`.

When the crawl is finished, commit the `data/` folder so that your raw data is backed up.

In [17]:
if STAGE not in {"trial", "second", "all"}:
    raise ValueError("HW3_STAGE must be trial, second, or all.")
selected_titles = job_titles[:1] if STAGE == "trial" else job_titles[1:2] if STAGE == "second" else job_titles
if RUN_LIVE:
    print(f"Stage: {STAGE}; {len(selected_titles)} titles, at most {3 * len(selected_titles)} pages.", flush=True)
    crawl_titles(selected_titles)
else:
    print("Offline mode: production crawl skipped. Set HW3_RUN_LIVE=1 explicitly to collect.")

Stage: all; 30 titles, at most 90 pages.


Cached: Data Scientist, page 1, 10 records
Cached: Data Scientist, page 2, 6 records
Cached: Data Scientist, page 3, 10 records
Cached: Machine Learning Engineer, page 1, 10 records
Cached: Machine Learning Engineer, page 2, 10 records
Cached: Machine Learning Engineer, page 3, 10 records
Rate limit: waiting 7 seconds before the next search.


Saved: Artificial Intelligence Specialist, page 1: 10 records -> data/artificial-intelligence-specialist-2026-10-30-21-11-01-205996-p1.json


   AI Specialist | SLAC
   Artificial Intelligence (AI) Specialist | Commonwealth Rolled Products, Inc.
   Artificial Intelligence Specialist | 631 Booz Allen Hamilton_United States
Rate limit: waiting 14 seconds before the next search.


Saved: Artificial Intelligence Specialist, page 2: 10 records -> data/artificial-intelligence-specialist-2026-10-30-21-11-18-211696-p2.json


...

   Modern Data & Strategy Architect | IBM
   Data Architect | W.C. Bradley Co.
   Enterprise Data Architect | Apex Systems
Rate limit: waiting 15 seconds before the next search.


Saved: Data Architect, page 2: 9 records -> data/data-architect-2026-10-30-21-34-15-767315-p2.json


   Enterprise Data Architect | Technosoft
   Data Architect - Investment Data & Analytics | Nashville, TN, USA | AllianceBernstein Holding L.P.
   Enterprise Data Architect (Data-Mesh) | Synechron
Rate limit: waiting 16 seconds before the next search.


Saved: Data Architect, page 3: 9 records -> data/data-architect-2026-10-30-21-34-32-777806-p3.json


   Sr. Architect, IT-Data Architecture | Baxter
   Data Architect - Insurance Data | Arthur J. Gallagher & Co.
   Data Architect - Modern Disability Claims | SteerBridge


## Collection audit and limitations

The table counts only newly collected responses and keeps the original job records intact. The instructor's example and offline test fixtures are excluded. An exhausted pagination chain may legitimately contain fewer than three pages; failures and incomplete titles must be reviewed before this part is considered finished.

A raw record can appear under multiple queries. Counting unique nonempty job IDs below is a collection diagnostic, not the cleaning step for HW-3.2. Results are a convenience sample of English-language U.S. searches during one collection window, rather than a representative census of the job market.

In [18]:
rows = crawl_summary(job_titles)
REPORT_DIR.mkdir(exist_ok=True)
with (REPORT_DIR / "crawl-summary.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["query", "pages_saved", "raw_records", "status"])
    writer.writeheader()
    writer.writerows(rows)

from IPython.display import HTML, display
header = "<tr>" + "".join(f"<th>{html.escape(key)}</th>" for key in rows[0]) + "</tr>"
body = "".join("<tr>" + "".join(f"<td>{html.escape(str(value))}</td>" for value in row.values()) + "</tr>" for row in rows)
display(HTML("<table>" + header + body + "</table>"))

pages = saved_pages()
records = [job for _, value in pages for job in value.get("jobs_results", [])]
job_ids = {job["job_id"] for job in records if job.get("job_id")}
print("Production response files:", len(pages))
print("Raw job records (before deduplication):", len(records))
print("Distinct nonempty job IDs:", len(job_ids))
print("Records without job_id:", sum(not job.get("job_id") for job in records))
print("Recorded search attempts (including failed attempts):", len(load_attempts()))
complete = all(row["status"] in {"complete_3_pages", "exhausted"} for row in rows)
print("Part 1 collection status:", "COMPLETE" if complete else "INCOMPLETE — real collection/review still required")
if pages:
    collected = sorted(value["crawl_metadata"]["collected_at_utc"] for _, value in pages)
    print("Collection window (UTC):", collected[0], "to", collected[-1])
    latest_file, latest_saved = max(pages, key=lambda item: item[1]["crawl_metadata"]["collected_at_utc"])
    read_back = json.loads(latest_file.read_text(encoding="utf-8"))
    assert len(read_back.get("jobs_results", [])) == latest_saved["crawl_metadata"]["job_count"]
    print("Final saved-file read-back check: passed")

query,pages_saved,raw_records,status
Data Scientist,3,26,complete_3_pages
Machine Learning Engineer,3,30,complete_3_pages
Artificial Intelligence Specialist,3,30,complete_3_pages
Data Analyst,3,29,complete_3_pages
Business Intelligence Analyst,3,27,complete_3_pages
Research Scientist (AI-ML),3,29,complete_3_pages
Deep Learning Engineer,3,30,complete_3_pages
NLP Engineer (Natural Language Processing),3,28,complete_3_pages
Computer Vision Engineer,3,26,complete_3_pages
Data Engineer,3,29,complete_3_pages


Production response files: 90
Raw job records (before deduplication): 847
Distinct nonempty job IDs: 847
Records without job_id: 0
Recorded search attempts (including failed attempts): 90
Part 1 collection status: COMPLETE
Collection window (UTC): 2026-10-30T21:09:19.175143+00:00 to 2026-10-30T21:34:32.777806+00:00
Final saved-file read-back check: passed


## Collection result

The live crawl completed on October 30, 2026. All **30 job titles** returned **three saved response pages**, producing **90 JSON files** and **847 raw job records**. Each title contributed between **24 and 30 records**. All records have a title, company, description, and nonempty job ID. The instructor's example is excluded from these counts.

The 847 records contain 847 distinct job IDs. Substantive duplicates, cross-platform postings, and inconsistent fields still need to be assessed in the later cleaning parts; the original responses have been preserved here.

All **90 search attempts succeeded**, without automatic retries or duplicate request fingerprints. Collection took approximately **25.2 minutes**. The live account reported a limit of **250 searches per hour**, so requests were paced at a minimum of **17 seconds** rather than relying on the starter's older 50-search/hour assumption. The measured minimum interval was **17.001 seconds**. The final account check showed **160 searches remaining**.

Validation included 14 passing offline tests, checking the saved pagination chain for every title, reading all response JSON files, checking record counts against the coverage CSV, confirming every notebook code cell ran without error, and scanning project text files for the active API key. See `reports/crawl-summary.csv`, `reports/search-attempts.json`, and `reports/collection-audit.json` for the collection evidence. Raw data is retained for the next parts; Git review, commit, and push are performed by the student.


## Reproducibility and submission

1. Install the official client and notebook dependencies with `python -m pip install -r requirements.txt`. Never install the legacy `google-search-results` package alongside `serpapi`.
2. Save the SerpApi key in `~/.api-keys.json`, or set `HW3_KEYS_FILE` to another JSON file **outside** this repository. Do not display the key or the full account response.
3. Run the offline tests first. Then run this notebook with `HW3_RUN_LIVE=1` and `HW3_STAGE=trial` for the first title, inspect the saved JSON, run `HW3_STAGE=second`, and inspect again. Finally use `HW3_STAGE=all` to resume all 30 titles. The interval is calculated from the live account throughput limit with a safety margin (75 seconds if the limit is unavailable). Runtime therefore depends on the account: approximately 25 minutes at a 17-second interval, or two hours at a 75-second interval. Run only one crawler process at a time, and avoid using the same account for other searches during the crawl.
4. The default remains offline. Successful responses are reused from disk, including partly completed title chains. Failed requests are recorded, consume the local attempt budget, and are not automatically retried. Rerunning explicitly retries missing pages. Corrupt JSON stops the run so it can be inspected rather than silently spending more credits.
5. Review `reports/crawl-summary.csv`, the notebook's completion status, and any failures. Save the executed notebook, then render using `quarto render HW-3.1-crawl.ipynb --no-execute`. The HTML embeds its resources and rendering cannot trigger searches.
6. Review the changes and commit the notebook, raw `data/`, collection reports, tests, dependency file, and HTML together. Git submission/push is performed by the student. The final `commit-history.txt` is exported after the later homework parts are finished.

References: [SerpApi Python client](https://serpapi.com/integrations/python), [Google Jobs pagination](https://serpapi.com/google-jobs-api), and [free account/quota endpoint](https://serpapi.com/account-api). Accessed October 30, 2026.